# Temporal automobile-insurance fraud evaluation

This notebook demonstrates the reconstruction workflow. Download `fraud_oracle.csv` from the cited Figshare source into `data/` before running.

In [ ]:
import pandas as pd
from fraud_portfolio.data import audit_frame, temporal_split
from fraud_portfolio.modeling import build_logistic_pipeline, candidate_configurations
from fraud_portfolio.evaluation import binary_metrics
from fraud_portfolio.thresholds import threshold_curve, select_threshold

df = pd.read_csv('../data/fraud_oracle.csv')
audit_frame(df)


In [ ]:
(X_train, y_train), (X_val, y_val), (X_test, y_test) = temporal_split(df)

candidates = []
for cfg in candidate_configurations():
    model = build_logistic_pipeline(X_train, **cfg)
    model.fit(X_train, y_train)
    scores = model.predict_proba(X_val)[:, 1]
    metrics = binary_metrics(y_val, scores)
    candidates.append((metrics['pr_auc'], cfg, model, scores))

_, best_cfg, best_model, val_scores = max(candidates, key=lambda x: x[0])
best_cfg


In [ ]:
curve = threshold_curve(y_val, val_scores)
threshold = select_threshold(curve, objective='f1')
test_scores = best_model.predict_proba(X_test)[:, 1]
binary_metrics(y_test, test_scores, threshold=threshold)


## Interpretation

Do not use overall accuracy alone for a rare-event problem. Threshold selection should ultimately follow business costs and investigation capacity, not an arbitrary metric objective.